# Feed Junk Head — `google/siglip2-base-patch16-naflex`

Trains a lightweight binary classifier (suitable vs junk) on top of frozen naflex embeddings, to keep
QR codes, UI screenshots, and scanned documents out of the global discovery feed.

**Why not an aesthetic scorer:** AVA-style aesthetic datasets contain only photography — QR codes and
screenshots are out-of-distribution for them, so a scorer gives them undefined (not reliably low)
scores. Junk is a *content-type* problem: train on explicit junk classes instead. A graded aesthetic
scorer can come later for ranking, once we have domain data.

**Datasets** (all ungated, sampled — nothing is downloaded in full):
- junk: `creative-graphic-design/Rico` (app UI screenshots), `chainyo/rvl-cdip` (scanned documents,
  forms, letters), synthetic QR codes generated in-notebook
- suitable: `detection-datasets/coco` (everyday photos), `huggan/wikiart` (art / illustration)

**Output:** `feed_junk_head.onnx` — input: float32[B, 768] L2-normalized embedding → output:
float32[B, 1] logit, **positive = junk**

**Runtime:** GPU (T4 recommended). Embedding ~30k images takes ~20 min; training is <1 min.

---
**Steps:**
1. Install deps & check GPU
2. Mount Drive (cache embeddings + ONNX across sessions)
3. Load frozen naflex backbone
4. Stream-sample the datasets & pre-compute embeddings (per-source cache)
5. Train MLP head
6. Evaluate (AUC, accuracy, per-source recall)
7. Export to ONNX & verify

## 1. Setup

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU — embedding step will be very slow")

In [ ]:
!pip install -q transformers datasets scikit-learn onnx onnxruntime pillow tqdm "qrcode[pil]"

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/currents_moderation"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Save directory:", SAVE_DIR)

## 3. Load Frozen Backbone

In [ ]:
from transformers import AutoModel, AutoProcessor
import torch
import torch.nn.functional as F

CHECKPOINT = "google/siglip2-base-patch16-naflex"
MAX_PATCHES = 256  # must match inference/main.py

processor = AutoProcessor.from_pretrained(CHECKPOINT)
backbone = AutoModel.from_pretrained(CHECKPOINT, torch_dtype=torch.float32)
backbone.eval()
for p in backbone.parameters():
    p.requires_grad = False

device = "cuda" if torch.cuda.is_available() else "cpu"
backbone = backbone.to(device)
print(f"Backbone loaded on {device}")

## 4. Stream-Sample Datasets & Pre-compute Embeddings

Each source is streamed (no full downloads), embedded in batches, and cached to Drive as its own
`.pt` file — a crash costs at most one source. The image column is detected dynamically, so config
or schema drift in the upstream datasets only needs a constant tweaked here.

In [ ]:
# Samples per source. Junk ≈ suitable overall (15k vs 15k).
N_RICO    = 6000   # app UI screenshots
N_RVL     = 6000   # scanned documents / forms / letters
N_QR      = 3000   # synthetic QR codes
N_COCO    = 7500   # everyday photos
N_WIKIART = 7500   # art / illustration

# RVL-CDIP is grayscale scans — convert this fraction of *suitable* samples to
# grayscale too, so the head can't learn the "grayscale ⇒ junk" shortcut.
GRAYSCALE_POSITIVE_FRACTION = 0.10

In [ ]:
from tqdm.auto import tqdm
from PIL import Image
import itertools
import random

random.seed(42)


def _find_image(row):
    """Return the first PIL image value in a dataset row."""
    for v in row.values():
        if hasattr(v, "convert"):
            return v
    raise KeyError(f"no image column in row with keys {list(row.keys())}")


def embed_pil_batches(images_iter, total, save_path, batch_size=32):
    """Embed an iterable of PIL images, caching the result to save_path."""
    if os.path.exists(save_path):
        print(f"cached: {save_path}")
        return torch.load(save_path)

    all_embs = []
    batch = []
    pbar = tqdm(total=total, desc=os.path.basename(save_path))

    def flush():
        nonlocal batch
        if not batch:
            return
        inputs = processor(images=batch, max_num_patches=MAX_PATCHES, return_tensors="pt").to(device)
        with torch.no_grad():
            feats = backbone.get_image_features(**inputs)
            if hasattr(feats, "pooler_output"):
                feats = feats.pooler_output
        all_embs.append(F.normalize(feats.cpu().float(), dim=-1))
        pbar.update(len(batch))
        batch = []

    for img in itertools.islice(images_iter, total):
        batch.append(img.convert("RGB") if img.mode != "RGB" else img)
        if len(batch) >= batch_size:
            flush()
    flush()
    pbar.close()

    embeddings = torch.cat(all_embs)
    torch.save(embeddings, save_path)
    return embeddings


def hf_images(dataset_id, n, config=None, split="train", grayscale_fraction=0.0):
    """Stream a Hugging Face dataset and yield its PIL images."""
    from datasets import load_dataset
    ds = load_dataset(dataset_id, config, split=split, streaming=True)
    ds = ds.shuffle(seed=42, buffer_size=2000)
    for row in itertools.islice(iter(ds), n):
        img = _find_image(row)
        if grayscale_fraction and random.random() < grayscale_fraction:
            img = img.convert("L")
        yield img

In [ ]:
import io
import string
import qrcode


def qr_images(n, background_images=None):
    """Synthetic QR codes: plain black/white, occasionally colored, and some
    pasted onto real photos (QR-in-a-photo is a common junk save shape)."""
    backgrounds = list(background_images) if background_images else []
    for i in range(n):
        payload = "https://" + "".join(random.choices(string.ascii_lowercase + string.digits, k=random.randint(8, 60)))
        qr = qrcode.QRCode(
            version=None,
            error_correction=random.choice([qrcode.constants.ERROR_CORRECT_L, qrcode.constants.ERROR_CORRECT_M, qrcode.constants.ERROR_CORRECT_H]),
            box_size=random.randint(4, 12),
            border=random.randint(1, 6),
        )
        qr.add_data(payload)
        qr.make(fit=True)
        if random.random() < 0.15:
            fill, back = random.choice([("#1d3557", "#f1faee"), ("#2a9d8f", "white"), ("black", "#ffe8d6"), ("#7b2cbf", "white")])
        else:
            fill, back = "black", "white"
        img = qr.make_image(fill_color=fill, back_color=back).convert("RGB")

        if backgrounds and random.random() < 0.3:
            bg = random.choice(backgrounds).convert("RGB").resize((640, 640))
            scale = random.uniform(0.35, 0.85)
            side = int(640 * scale)
            qr_small = img.resize((side, side))
            x = random.randint(0, 640 - side)
            y = random.randint(0, 640 - side)
            bg.paste(qr_small, (x, y))
            img = bg
        yield img

In [ ]:
# Embed every source (each cached independently on Drive).
# label 1 = junk, 0 = suitable

coco_iter_for_qr = hf_images("detection-datasets/coco", 200)
qr_backgrounds = list(coco_iter_for_qr)

SOURCES = [
    # (name, label, embeddings)
    ("rico", 1, lambda: hf_images("creative-graphic-design/Rico", N_RICO)),
    ("rvl", 1, lambda: hf_images("chainyo/rvl-cdip", N_RVL)),
    ("qr", 1, lambda: qr_images(N_QR, qr_backgrounds)),
    ("coco", 0, lambda: hf_images("detection-datasets/coco", N_COCO, grayscale_fraction=GRAYSCALE_POSITIVE_FRACTION)),
    ("wikiart", 0, lambda: hf_images("huggan/wikiart", N_WIKIART, grayscale_fraction=GRAYSCALE_POSITIVE_FRACTION)),
]
COUNTS = {"rico": N_RICO, "rvl": N_RVL, "qr": N_QR, "coco": N_COCO, "wikiart": N_WIKIART}

embs, lbls, srcs = [], [], []
for name, label, images in SOURCES:
    e = embed_pil_batches(images(), COUNTS[name], f"{SAVE_DIR}/embeddings_junk_{name}.pt")
    embs.append(e)
    lbls.append(torch.full((len(e),), float(label)))
    srcs.extend([name] * len(e))

embeddings = torch.cat(embs)
labels = torch.cat(lbls)
import numpy as np
sources = np.array(srcs)
print(f"Embeddings: {embeddings.shape}, junk: {int(labels.sum())}, suitable: {int((1 - labels).sum())}")

## 5. Train MLP Head

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn

# 80/20 split, stratified by source so val covers every junk type.
g = torch.Generator().manual_seed(42)
perm = torch.randperm(len(embeddings), generator=g)
val_mask = torch.zeros(len(embeddings), dtype=torch.bool)
for name in COUNTS:
    idx = perm[torch.tensor(sources == name)[perm]]
    val_mask[idx[: int(len(idx) * 0.2)]] = True

train_ds = TensorDataset(embeddings[~val_mask], labels[~val_mask])
val_ds   = TensorDataset(embeddings[val_mask], labels[val_mask])
val_sources = sources[val_mask.numpy()]

train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=1024, shuffle=False)
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

In [ ]:
class JunkHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        return self.net(x)


head = JunkHead().to(device)
optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=0.01)
criterion = nn.BCEWithLogitsLoss()

EPOCHS = 20
best_val_loss = float("inf")
best_state = None

for epoch in range(1, EPOCHS + 1):
    head.train()
    train_loss = 0.0
    for emb, lbl in train_loader:
        emb, lbl = emb.to(device), lbl.to(device)
        optimizer.zero_grad()
        loss = criterion(head(emb).squeeze(1), lbl)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(emb)
    train_loss /= len(train_ds)

    head.eval()
    val_loss = 0.0
    with torch.no_grad():
        for emb, lbl in val_loader:
            emb, lbl = emb.to(device), lbl.to(device)
            val_loss += criterion(head(emb).squeeze(1), lbl).item() * len(emb)
    val_loss /= len(val_ds)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in head.state_dict().items()}

    print(f"Epoch {epoch:02d}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}")

head.load_state_dict(best_state)
print(f"\nBest val loss: {best_val_loss:.4f}")

## 6. Evaluate

Overall AUC plus per-source recall — the junk sources tell us whether each failure mode
(screenshots / documents / QR codes) is actually caught; the suitable sources tell us the
false-positive rate on legit content.

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

head.eval()
all_logits, all_labels = [], []
with torch.no_grad():
    for emb, lbl in val_loader:
        all_logits.append(head(emb.to(device)).squeeze(1).cpu())
        all_labels.append(lbl)

all_logits = torch.cat(all_logits).numpy()
all_labels = torch.cat(all_labels).numpy()
all_probs  = 1 / (1 + np.exp(-all_logits))
all_preds  = (all_probs >= 0.5).astype(int)

print(f"AUC-ROC:  {roc_auc_score(all_labels, all_probs):.4f}")
print(f"Accuracy: {accuracy_score(all_labels, all_preds):.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=["suitable", "junk"]))

print("Per-source accuracy on val:")
for name in COUNTS:
    mask = val_sources == name
    if mask.sum() == 0:
        continue
    acc = accuracy_score(all_labels[mask], all_preds[mask])
    print(f"  {name:8s} n={int(mask.sum()):5d}  acc={acc:.4f}")

## 7. Export to ONNX & Verify

In [ ]:
import onnx
import onnxruntime as ort

ONNX_PATH = f"{SAVE_DIR}/feed_junk_head.onnx"

head_cpu = head.cpu()
head_cpu.eval()
dummy_input = torch.randn(1, 768)

torch.onnx.export(
    head_cpu,
    dummy_input,
    ONNX_PATH,
    input_names=["embedding"],
    output_names=["logits"],
    dynamic_axes={"embedding": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)

onnx.checker.check_model(ONNX_PATH)
print(f"ONNX model saved and validated: {ONNX_PATH}")

In [ ]:
# Verify ONNX output matches PyTorch within tolerance
session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])

test_emb = embeddings[:16].numpy()
with torch.no_grad():
    pt_logits = head_cpu(torch.from_numpy(test_emb)).squeeze(1).numpy()
ort_logits = session.run(None, {"embedding": test_emb})[0].squeeze(1)

max_diff = np.abs(pt_logits - ort_logits).max()
print(f"Max PyTorch vs ONNX logit difference: {max_diff:.2e}")
assert max_diff < 1e-4, f"ONNX mismatch too large: {max_diff}"
print("ONNX output verified ✓")
print(f"\nModel ready at: {ONNX_PATH}")